## 第10章　导数 —— 斜率的"瞬时版本"

> 本章目标：掌握解析导数的速查公式（多项式、指数、sigmoid、ReLU），并用数值微分验证每一个公式。理解二阶导数的含义——变化的加速度——以及 ReLU 在 x=0 处"不可导"对深度学习意味着什么。
> 前置知识：第 2 章（割线→切线的几何直觉）、第 1 章（函数、NumPy）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")



### 10.1　数值微分 —— 用代码"测量"导数

第 2 章你亲手验证了"缩小区间 → 割线斜率趋近于切线斜率"。现在把这个过程固化为一个通用工具——**数值微分**：给定任意函数 f 和任意点 x，自动算出该点的导数近似值。

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

这是**中心差分公式**——比单侧差分 `(f(x+h)−f(x))/h` 精确得多（误差从 O(h) 降到 O(h²)）。它不关心 f 是什么函数——多项式、指数、三角函数、甚至你自定义的任何 Python 函数——都能直接"测量"出导数。

但这个工具有一个致命弱点，你在第 2 章已经亲眼见证过：**h 太大 → 近似粗糙；h 太小 → 浮点舍入误差主导，结果崩溃。** 这就是为什么 PyTorch 用自动微分而不是数值微分做训练——但数值微分在**验证**自动微分是否正确时（梯度检验，第 28 章）是不可替代的。

📐 **定义　数值微分（Numerical Differentiation）**：f'(x) ≈ (f(x+h) − f(x−h)) / (2h)。h 通常在 1e-5 到 1e-8 之间选取。不是精确计算，而是一个可控制的近似——精度和稳定性的权衡由 h 决定。

💻 **代码　通用数值微分工具 + 精度 vs h 的关系验证**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    """中心差分法：对任意函数 f 在点 x 处求数值导数"""
    return (f(x + h) - f(x - h)) / (2 * h)

# 测试：f(x) = x², f'(x) = 2x, 在 x=3 处应为 6
def f_square(x):
    return x ** 2

x0 = 3.0
nd = numerical_derivative(f_square, x0)
exact = 2 * x0  # 解析导数
print(f"f(x)=x² 在 x={x0}: 数值={nd:.6f}, 解析={exact}, 误差={abs(nd-exact):.2e}")

# 精度 vs h 的扫描
print(f"\n{'h':<14} {'数值导数':<16} {'误差':<12}")
print("-" * 42)
for h in [1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13]:
    nd_h = numerical_derivative(f_square, x0, h)
    err = abs(nd_h - exact)
    print(f"h={h:<10.0e}  {nd_h:<16.12f}  {err:<12.2e}")
print("\n最优精度在 h=1e-5 ~ 1e-7；h<1e-11 时舍入误差开始主导")




> **关键洞察**：数值微分 = 导数的"万用表"——不挑函数、不需要公式，当场测量。在深度学习研究中，每当你手写了一个新的 autograd 操作（第 28 章），第一件事就是用数值微分验证它的梯度是否正确——这叫"梯度检验"（gradient check），是反向传播实现中最重要的调试手段。

🔗 **AI 连接**：数值微分在第 28 章手写反向传播时用于验证手工梯度；在第 23 章梯度裁剪时帮助理解"梯度多大算太大"。PyTorch 的 `torch.autograd.gradcheck` 本质上就是自动化版的 `numerical_derivative`。

---

### 10.2　验证幂函数 —— x³ 在 x=2 处导数接近 12

数值微分是"测量"，解析导数是"公式"。先用最简单的幂函数建立信心：f(x)=x³，理论导数 f'(x)=3x²。在 x=2 处，数值微分和解析导数应该高度一致。

📐 **定义　解析导数（Analytical Derivative）**：通过求导法则推导出的精确导数公式。与数值微分的"近似"不同，解析导数是**精确**的——不依赖 h 的选择。幂法则：xⁿ → n·xⁿ⁻¹。

💻 **代码　验证 x³ 和 eˣ 的导数**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

# ===== x³ —— 幂函数 =====
def cubic(x): return x ** 3
def cubic_deriv(x): return 3 * x ** 2

# ===== eˣ —— 导数等于自身！ =====
def exp_func(x): return np.exp(x)
def exp_deriv(x): return np.exp(x)

for name, f, fd, x0, expected in [
    ("x³ 在 x=2", cubic, cubic_deriv, 2.0, 12.0),
    ("eˣ 在 x=1", exp_func, exp_deriv, 1.0, np.exp(1.0)),
]:
    nd = numerical_derivative(f, x0)
    ad = fd(x0)
    print(f"{name}: 数值={nd:.6f}, 解析={ad:.6f}, 匹配={abs(nd-ad)<1e-5}")




> **关键洞察**：幂函数 x³ 的导数 3x² 再简单不过——但亲眼看到数值微分以 1e-5 的精度验证了这个公式，你就建立了"数值微分可以验证任何导数"的信心。

---

### 10.3　验证指数函数 —— eˣ 的导数等于自身

在所有函数中，eˣ 有一个独一无二的性质：**它的导数就是它自己。** f(x)=eˣ → f'(x)=eˣ。这意味着在 x=1 处，函数值和导数值完全相同（都等于 e≈2.718）。

这个性质让 eˣ 在微积分和深度学习中反复出现——softmax 的 exp、sigmoid 内部的 exp、损失函数的对数抵消——全都依赖于 eˣ 的"自我复制"能力。

💻 **代码　sigmoid 和 ReLU 的导数验证**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

# sigmoid
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# ReLU
def relu(x): return np.maximum(0, x)
def relu_deriv(x):
    return np.where(x > 0, 1.0, 0.0)

for name, f, fd, x0 in [
    ("sigmoid x=0", sigmoid, sigmoid_deriv, 0.0),
    ("ReLU x=3", relu, relu_deriv, 3.0),
    ("ReLU x=-2", relu, relu_deriv, -2.0),
]:
    nd = numerical_derivative(f, x0)
    ad = fd(x0)
    print(f"{name}: 数值={nd:.6f}, 解析={ad:.6f}, 匹配={abs(nd-ad)<1e-5}")




> **关键洞察**：**sigmoid 的导数** σ(x)(1−σ(x)) 是全书最重要的导数公式之一——它表明 sigmoid 在 x 远离 0 时导数趋近于 0（饱和），这就是为什么深层网络用 sigmoid 会导致梯度消失（第 23 章）。

---

### 10.4　数值微分的通用性 —— 建立直觉与验证的利器

数值微分最强大的地方不在于算某个具体函数的导数，而在于它是**通用的**——给定任意函数 f 和任意点 x，不需要知道 f 的解析形式，就能直接"测量"出导数值。

这个通用性在深度学习研究中至关重要：每当你手写了一个新的 autograd 操作（第 28 章），第一件事就是用数值微分验证它的梯度是否正确——这叫"梯度检验"（gradient check）。

| 函数 f(x) | 导数 f'(x) | 直觉 |
|-----------|-----------|------|
| eˣ | eˣ | 唯一导数等于自身的函数 |
| ln(x) | 1/x | x 越大增长越慢 |
| tanh(x) | 1 − tanh²(x) | 类似 sigmoid 但值域 (−1, 1) |
| sin(x) | cos(x) | 相位偏移 π/2 |
| cos(x) | −sin(x) | 导数回到 sin，取负 |

🔗 **AI 连接**：sigmoid 导数在第 27 章逻辑回归反向传播中起核心作用；ReLU 导数在第 28 章两层网络反向传播中决定哪些神经元"活着"（梯度非零）；第 13 章链式法则会把这些导数组装成反向传播的梯度链条。

---

### 10.5　二阶导数 —— "变化的变化"

一阶导数 f'(x) 告诉你"函数在上升还是下降、有多快"。二阶导数 f''(x) 告诉你**"变化本身在加速还是减速"**——即一阶导数的变化率。

- f''(x) > 0：斜率在增大 → 曲线向上弯（**凸** / convex）→ 就像踩油门加速
- f''(x) < 0：斜率在减小 → 曲线向下弯（**凹** / concave）→ 就像踩刹车减速
- f''(x) = 0：斜率不变 → 此处是**拐点**（inflection point）

在深度学习中，二阶导数最重要的应用是：**判断一个临界点（梯度 = 0 的地方）是最小值、最大值还是鞍点。** 优化器中的动量方法（第 24 章）和 Newton 方法都利用了二阶导数信息来加速收敛。

📐 **定义　二阶导数（Second Derivative）**：f''(x) = (f')'(x)——导数的导数。数值近似：f''(x) ≈ (f(x+h) − 2f(x) + f(x−h)) / h²。

💻 **代码　二阶导数数值计算：对比 x²、x³、sin(x) 的凹凸性**




In [ ]:
import numpy as np

def numerical_second_derivative(f, x, h=1e-4):
    """中心二阶差分：f''(x) ≈ (f(x+h) - 2f(x) + f(x-h)) / h²"""
    return (f(x + h) - 2 * f(x) + f(x - h)) / (h ** 2)

# 测试函数
funcs = {
    "x²":    (lambda x: x**2,    lambda x: 2.0),          # f''=2 > 0 → 永远是凸的
    "x³":    (lambda x: x**3,    lambda x: 6 * x),         # f''=6x → x>0凸, x<0凹
    "sin(x)":(lambda x: np.sin(x), lambda x: -np.sin(x)),  # f''=-sin → 周期性凹凸
}

print(f"{'函数':<10} {'x':<6} {'数值 f\"(x)':<14} {'解析 f\"(x)':<14} {'凹凸性'}")
print("-" * 58)
for name, (f, f_dd) in funcs.items():
    for x in [-2.0, 0.0, 2.0]:
        nd2 = numerical_second_derivative(f, x)
        ad2 = f_dd(x)
        if ad2 > 0.01:
            shape = "凸 (∪)"
        elif ad2 < -0.01:
            shape = "凹 (∩)"
        else:
            shape = "拐点"
        print(f"{name:<10} {x:<6.1f} {nd2:<14.6f} {ad2:<14.6f} {shape}")

print("\n观察: x² 永远凸(∪)——梯度下降若损失函数是凸函数则保证收敛")
print("      x³ 在 x<0 时凹(∩), x>0 时凸(∪)——x=0 是拐点")




> **关键洞察**：凸函数（f''>0 处处成立）是优化理论中的"好函数"——任何局部最小值就是全局最小值，梯度下降保证收敛。线性回归的 MSE 损失就是凸函数（第 26 章）。但神经网络的损失函数是高度非凸的（有无数个局部最小值和鞍点）——这就是为什么优化器需要 Momentum 和 Adam（第 24 章）来逃离鞍点和浅谷。

🔗 **AI 连接**：二阶导数 → 损失函数曲率 → 学习率的上限。曲率越大（f'' 越大），学习率必须越小，否则梯度下降会"冲过头"（第 24 章）。第 23 章梯度裁剪本质上是用一阶近似来应对二阶曲率过大的场景。

---

### 10.6　ReLU 在 x=0 处的"不可导"——理论缺陷 vs 工程现实

ReLU 的导数在 x=0 处**数学上不存在**（左导数为 0，右导数为 1，不相等）。但在深度学习中，这从来不是一个问题——原因有三：

1. **数值上**：x=0 恰好出现的概率在浮点数下几乎为 0（连续空间中的一个点测度为 0）
2. **工程上**：PyTorch 的 `ReLU` 在 x=0 处返回导数为 0（`gradient = 0`），称为"subgradient"——任意选择左或右导数之一，实践中完全不影响训练
3. **数学上**：ReLU 是"几乎处处可导"的——除 x=0 这一个点外处处可导，而 Lebesgue 积分不关心单点的行为

**真正需要警惕的不是 ReLU 在 x=0 的不可导，而是 ReLU 在 x<0 时导数恒为 0——导致"死亡 ReLU"问题：一旦某个神经元对所有输入都输出负值，它的梯度永远为 0，权重永远不更新，这个神经元就"死"了。** LeakyReLU 和 GELU 正是为解决这个问题而设计的。

💻 **代码　亲眼观察 ReLU 在 x=0 附近的行为**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

def relu(x):
    return np.maximum(0, x)

# 围绕 x=0 仔细观察
print(f"{'x':<12} {'ReLU(x)':<12} {'数值导数':<14} {'PyTorch风格':<14}")
print("-" * 52)
for x in [-0.1, -1e-4, -1e-8, 0.0, 1e-8, 1e-4, 0.1]:
    nd = numerical_derivative(relu, x, h=1e-6)
    pt_style = 1.0 if x > 0 else 0.0  # PyTorch/subgradient 约定
    print(f"x={x:<10.6f} {relu(x):<12.6f} {nd:<14.6f} {pt_style:<14.1f}")

print("\n观察: x 从负侧逼近 0 → 导数为 0")
print("      x 从正侧逼近 0 → 导数为 1")
print("      x 恰好 = 0 时数值微分给 ~0.5（中心差分跨过不连续点）")
print("      PyTorch 选择在 x=0 处返回 0（subgradient）——实践中完全不影响训练")




> **关键洞察**：ReLU 在 x=0 处的"不可导"是一个教科书喜欢强调、但工程上从不担心的细节。深度学习框架选择一个 subgradient 值（通常 0），训练照样顺利进行。这揭示了一个更大的原则：**在深度学习中，"几乎处处可导"就足够了——你不需要完美的数学优雅，你需要的是计算效率和梯度流通。**

🔗 **AI 连接**：第 28 章手写反向传播时，ReLU 的反向传播只用一行 `grad[input <= 0] = 0`；第 25 章 Kaiming 初始化专门为 ReLU 设计——因为 ReLU 把一半的输入清零了，初始化需要补偿这种"信息减半"。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）数值微分的中心差分公式是什么？h 选 1e-3 和 1e-12 哪个更准确？为什么？
2. （概念）写出 sigmoid 函数 σ(x) 的导数公式。当 x 很大（如 10）或很小（如 −10）时，σ'(x) 接近多少？这个现象在深度学习中叫什么问题？
3. （概念）ReLU 在 x=0 处数学上不可导，为什么在深度学习中这从来不是一个问题？真正需要担心的"ReLU 死亡"是什么意思？
4. （代码）用数值微分验证 tanh(x) 在 x=0 处的导数（解析导数 = 1 − tanh²(0) = 1）。对 f(x)=ln(x) 在 x=2 处做同样的验证（解析导数 = 1/2）。
5. （代码）对 f(x)=x³ − 3x 求一阶导数和二阶导数。找到 f'(x)=0 的所有点（临界点），用二阶导数判断每个临界点是极大值、极小值还是拐点。画出 f(x) 及其导数验证你的判断。
---


> 🔗 **章末钩子**：你现在能求任何一个单变量函数的导数了。但神经网络有上亿个参数——损失函数不是一元函数，而是多元函数。对一个参数求导时，其他参数怎么办？
> 预览：下一章——**偏导数**，按住一个，动另一个。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
